# 02_preprocessing.ipynb — Preprocesamiento y Splits (Dataset: IMDB)

Este notebook toma los datos crudos del dataset IMDB, los limpia, los transforma en representaciones numéricas y genera los splits listos para entrenar. Al final exporta los datos procesados para que cada notebook de modelo los cargue directamente sin repetir trabajo.

La estructura es la siguiente:

1. Imports y configuración
2. Carga y estandarización de IMDB
3. Limpieza de texto
4. Split estratificado (train / val / test)
5. Vectorización TF-IDF (para MLP)
6. Tokenización con secuencias (para LSTM)
7. Tokenización con DistilBERT (para Transformer)
8. Guardado de todos los artefactos
9. Verificación final

## CELDA 1 — Instalación

In [5]:
!pip install pandas numpy scikit-learn nltk transformers torch joblib tensorflow scipy

     ---------------------------------------- 0.0/350.6 MB ? eta -:--:--
     ---------------------------------------- 0.1/350.6 MB 3.2 MB/s eta 0:01:50
     ---------------------------------------- 0.3/350.6 MB 3.4 MB/s eta 0:01:44
     ---------------------------------------- 0.6/350.6 MB 5.0 MB/s eta 0:01:10
     ---------------------------------------- 1.2/350.6 MB 7.5 MB/s eta 0:00:47
     ---------------------------------------- 1.4/350.6 MB 6.4 MB/s eta 0:00:55
     ---------------------------------------- 2.0/350.6 MB 8.0 MB/s eta 0:00:44
     ---------------------------------------- 2.5/350.6 MB 8.0 MB/s eta 0:00:44
     ---------------------------------------- 2.7/350.6 MB 7.5 MB/s eta 0:00:47
     ---------------------------------------- 3.2/350.6 MB 8.2 MB/s eta 0:00:43
     ---------------------------------------- 4.2/350.6 MB 9.5 MB/s eta 0:00:37
      -------------------------------------- 4.8/350.6 MB 10.0 MB/s eta 0:00:35
      --------------------------------------- 5


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## CELDA 2 — Imports

In [6]:
import pandas as pd
import numpy as np
import re
import os
import joblib
import nltk
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import DistilBertTokenizer

nltk.download('stopwords')
STOPWORDS_EN = set(stopwords.words('english'))

# Carpeta donde se guardan los artefactos procesados
os.makedirs('../data/processed', exist_ok=True)

print("Librerias cargadas correctamente")


[TensorFlow DLL Diagnostic] Analyzing: c:\Users\luisa\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "c:\Users\luisa\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

## CELDA 3 — Carga y estandarización

Este bloque es igual al EDA pero aqui es donde queda la version definitiva limpia que se usa en todo el proyecto.

In [ ]:
# Cargar dataset IMDB
df = pd.read_csv('../data/imdb/IMDB Dataset.csv')
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df['text'] = df['review']
df = df[['text', 'label']].dropna().reset_index(drop=True)

print(f"IMDB Dataset: {df.shape}")
print(f"Distribucion de clases:")
print(df['label'].value_counts())

## CELDA 4 — Funcion de limpieza de texto

Esta funcion se aplica a los textos que van a MLP y LSTM. Para DistilBERT NO se aplica porque el modelo fue entrenado con texto casi crudo y necesita ver el lenguaje natural tal como es.

In [ ]:
def limpiar_texto(texto):
    # 1. Eliminar tags HTML (<br />, <a href=...>, etc.)
    texto = re.sub(r'<[^>]+>', ' ', texto)
    
    # 2. Eliminar URLs
    texto = re.sub(r'http\S+|www\S+', ' ', texto)
    
    # 3. Convertir a minusculas
    texto = texto.lower()
    
    # 4. Eliminar caracteres que no son letras ni espacios
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    
    # 5. Eliminar espacios multiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    # 6. Eliminar stopwords
    palabras = texto.split()
    palabras = [p for p in palabras if p not in STOPWORDS_EN]
    
    return ' '.join(palabras)


# Aplicar limpieza al dataset IMDB
print("Limpiando IMDB...")
df['text_clean'] = df['text'].apply(limpiar_texto)

print("Limpieza completada")

# Verificacion rapida
print("\nEjemplo IMDB — original:")
print(df['text'].iloc[0][:200])
print("\nEjemplo IMDB — limpio:")
print(df['text_clean'].iloc[0][:200])

## CELDA 5 — Split estratificado

In [ ]:
def hacer_splits(df, columna_texto, columna_label, nombre):
    X = df[columna_texto].values
    y = df[columna_label].values
    
    # Primer corte: 85% train+val / 15% test
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y,
        test_size=0.15,
        random_state=42,
        stratify=y
    )
    
    # Segundo corte: del 85%, ~17.6% es val → queda 70/15/15 global
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size=0.176,
        random_state=42,
        stratify=y_trainval
    )
    
    print(f"\n{nombre}:")
    print(f"  Train: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
    print(f"  Val:   {len(X_val):,}   ({len(X_val)/len(X)*100:.1f}%)")
    print(f"  Test:  {len(X_test):,}  ({len(X_test)/len(X)*100:.1f}%)")
    
    # Verificar que la distribucion de clases se mantiene en cada split
    for nombre_split, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
        pos = y_split.sum() / len(y_split) * 100
        print(f"  {nombre_split} — positivos: {pos:.1f}%")
    
    return X_train, X_val, X_test, y_train, y_val, y_test


# Splits con texto limpio (para MLP y LSTM)
X_train, X_val, X_test, y_train, y_val, y_test = hacer_splits(
    df, 'text_clean', 'label', 'IMDB (texto limpio)'
)

# Splits con texto original (para DistilBERT)
X_train_raw, X_val_raw, X_test_raw, _, _, _ = hacer_splits(
    df, 'text', 'label', 'IMDB (texto raw para BERT)'
)

## CELDA 6 — Vectorizacion TF-IDF para MLP

In [ ]:
# Se entrena SOLO sobre train para no contaminar val y test
# max_features=50000 captura el vocabulario mas relevante sin explotar memoria

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)
tfidf.fit(X_train)

X_train_tfidf = tfidf.transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print("TF-IDF IMDB — shape train:", X_train_tfidf.shape)

# Guardar vectorizador
joblib.dump(tfidf, '../data/processed/tfidf_imdb.pkl')
print("Vectorizador TF-IDF guardado")

## CELDA 7 — Tokenizacion para LSTM

In [ ]:
# Parametros basados en el percentil 95 que viste en el EDA
MAX_WORDS = 20000   # tamano del vocabulario
MAX_LEN = 400       # ajustar segun percentil 95 de IMDB del EDA

# IMDB
tok = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tok.fit_on_texts(X_train)

seq_train = tok.texts_to_sequences(X_train)
seq_val   = tok.texts_to_sequences(X_val)
seq_test  = tok.texts_to_sequences(X_test)

X_train_seq = pad_sequences(seq_train, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_seq   = pad_sequences(seq_val,   maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq  = pad_sequences(seq_test,  maxlen=MAX_LEN, padding='post', truncating='post')

print("Secuencias IMDB — shape train:", X_train_seq.shape)

# Guardar tokenizador
joblib.dump(tok, '../data/processed/tokenizer_imdb.pkl')
print("Tokenizador LSTM guardado")

## CELDA 8 — Tokenizacion para DistilBERT

In [ ]:
bert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

MAX_LEN_BERT = 256  # DistilBERT acepta hasta 512 tokens, 256 es buen balance memoria/rendimiento

def tokenizar_bert(textos, tokenizer, max_len):
    return tokenizer(
        list(textos),
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'   # retorna tensores de PyTorch
    )

print("Tokenizando IMDB para DistilBERT...")
bert_train = tokenizar_bert(X_train_raw, bert_tokenizer, MAX_LEN_BERT)
bert_val   = tokenizar_bert(X_val_raw,   bert_tokenizer, MAX_LEN_BERT)
bert_test  = tokenizar_bert(X_test_raw,  bert_tokenizer, MAX_LEN_BERT)

print("Shapes (input_ids):")
print("  IMDB train:", bert_train['input_ids'].shape)

## CELDA 9 — Guardado de todos los artefactos

In [ ]:
import numpy as np

# Labels como arrays de numpy para todos los splits
np.save('../data/processed/y_train_imdb.npy', y_train)
np.save('../data/processed/y_val_imdb.npy',   y_val)
np.save('../data/processed/y_test_imdb.npy',  y_test)

# TF-IDF (matrices sparse)
from scipy import sparse
sparse.save_npz('../data/processed/tfidf_train_imdb.npz', X_train_tfidf)
sparse.save_npz('../data/processed/tfidf_val_imdb.npz',   X_val_tfidf)
sparse.save_npz('../data/processed/tfidf_test_imdb.npz',  X_test_tfidf)

# Secuencias LSTM
np.save('../data/processed/seq_train_imdb.npy', X_train_seq)
np.save('../data/processed/seq_val_imdb.npy',   X_val_seq)
np.save('../data/processed/seq_test_imdb.npy',  X_test_seq)

# Tokens BERT
import torch
torch.save(bert_train, '../data/processed/bert_train_imdb.pt')
torch.save(bert_val,   '../data/processed/bert_val_imdb.pt')
torch.save(bert_test,  '../data/processed/bert_test_imdb.pt')

print("Todos los artefactos guardados en ../data/processed/")

## CELDA 10 — Verificacion final

In [ ]:
from scipy import sparse

print("VERIFICACION FINAL DE ARTEFACTOS")
print("=" * 50)

# TF-IDF
tfidf_check = sparse.load_npz('../data/processed/tfidf_train_imdb.npz')
print(f"TF-IDF IMDB train:   {tfidf_check.shape}")

# Secuencias
seq_check = np.load('../data/processed/seq_train_imdb.npy')
print(f"Secuencias IMDB train: {seq_check.shape}")

# Labels
y_check = np.load('../data/processed/y_train_imdb.npy')
print(f"Labels IMDB train:   {y_check.shape} — positivos: {y_check.mean()*100:.1f}%")

# BERT
bert_check = torch.load('../data/processed/bert_train_imdb.pt')
print(f"BERT IMDB train input_ids: {bert_check['input_ids'].shape}")

print("\nTodo correcto. El preprocesamiento esta completo.")
print("Los notebooks de modelos pueden cargar los datos desde ../data/processed/")

## Resumen de lo que genera este notebook

```
data/processed/
├── tfidf_amazon.pkl          <- vectorizador TF-IDF entrenado (Amazon)
├── tfidf_imdb.pkl            <- vectorizador TF-IDF entrenado (IMDB)
├── tokenizer_amazon.pkl      <- tokenizador Keras para LSTM (Amazon)
├── tokenizer_imdb.pkl        <- tokenizador Keras para LSTM (IMDB)
│
├── tfidf_train/val/test_amazon.npz   <- matrices TF-IDF
├── tfidf_train/val/test_imdb.npz
│
├── seq_train/val/test_amazon.npy     <- secuencias para LSTM
├── seq_train/val/test_imdb.npy
│
├── bert_train/val/test_amazon.pt     <- tokens para DistilBERT
├── bert_train/val/test_imdb.pt
│
├── y_train/val/test_amazon.npy       <- labels
└── y_train/val/test_imdb.npy
```

Cuando termines de correrlo y confirmes que la verificacion final muestra los shapes correctos, avanzamos con el Paso 3: el modelo MLP.